## DMR generation from kerchunk


requires pydap, numpy installed

```python
pip install pydap
```


In [ ]:
import pydap
import json
# import numpy as np
from pydap.model import DatasetType
from pydap.responses.dmr import DMRResponse
from pydap.parsers.dmr import DummyData
from pathlib import Path

In [ ]:
directory_path = Path("./output_files/")

# Find all .json files in the top-level directory
json_files = list(directory_path.glob("*.json"))

filename = json_files[3]
filename

In [ ]:
with open(filename, "r") as file:
    data = json.load(file)

In [ ]:
data.keys()

In [ ]:
json.loads(data['.zattrs'])

In [ ]:
data['0.1.0']

In [ ]:
zarray = json.loads(data['.zarray'])
zarray

## Get metadata for pydap dataset

In [ ]:
zattrs = json.loads(data['.zattrs'])
zattrs 

## parse attribute metadata to fill in the pydap dataset

In [ ]:
zattrs = json.loads(data['.zattrs'])
dimensions = zattrs.pop("_ARRAY_DIMENSIONS")

ncattrs = dict((k,v) for k,v in zattrs.items() if k.isupper() and not k.startswith("NC_GLOBAL"))
_ = [zattrs.pop(key) for key in ncattrs]

global_attrs = [key for key in zattrs if "NC_GLOBAL" in key] + [key for key in zattrs if "#" not in key and not key.isupper()]

variables = set([key.split("#")[0] for key in zattrs if "#" in key and "NC_GLOBAL" not in key])
nc_varname = ncattrs.pop('NETCDF_VARNAME', None)


## global attributes at root level

In [ ]:
global_attributes = dict([(key.split("#")[-1], zattrs[key]) for key in global_attrs])
_ = [zattrs.pop(key) for key in global_attrs]
global_attributes

## variable attributes

In [ ]:
Variables = {}
for var in variables:
    attrs = dict((key.split("#")[-1], zattrs[key]) for key in zattrs if key.startswith(var))
    Variables.update({var: attrs})
Variables

In [ ]:
dim_shapes = dict(("/"+dim.lower(),size) for dim,size in zip(dimensions, zarray['shape']))
dim_shapes

### create pydap dataset 

It can generate a dap4-schema conforming dmr

In [ ]:
name = filename.name.replace("json","tif")
pyds = DatasetType(name = name, 
                   dimensions=dim_shapes,
                   attributes=global_attributes)
for var in variables:
    if var in nc_varname:
        data = DummyData(dtype=zarray['dtype'], shape=zarray['shape'], path='/')
        dims = [dim.lower() for dim in dimensions]
    else:
        if var in [dim.lower() for dim in dimensions]:
            data = DummyData(dtype='<i8', shape=(dim_shapes['/'+var],), path='/')
            dims = [var]
        else:
            data = None
            dims = None
    pyds.createVariable(name=var, data=data, dims=dims, **Variables[var])

In [ ]:
pyds.tree()

## store dmr in a local file 

It will append a .dmr to the filename (see below)

In [ ]:
dmr_name = f"./output_files/{name}.dmr"
dmr_name

In [ ]:
with open(dmr_name, "wb") as file:
    file.write(b"".join(DMRResponse(pyds)).decode("ascii").encode("utf-8"))